## Model Evaluation

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.nn.functional import softmax
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from torchvision import transforms
import torchvision.models.video as video_models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import random
import gc


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


DATA_DIR = "/kaggle/input/hwid12-highway-incidents-detection-dataset/Video-Accident-Dataset"
modelA_path = "/kaggle/input/incident-assessment/pytorch/default/1/models/subteam-a/two_stream_i3d_fast.pth"
modelB_path = "/kaggle/input/incident-assessment/pytorch/default/1/models/subteam-b/accident_detection_transformer.pth"

SEQ_LEN = 16
MAX_CLIPS_A = 8


class TwoStreamI3D_Fast(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_net = video_models.r3d_18(weights=None)
        self.flow_net = video_models.r3d_18(weights=None)
        feat_dim = self.rgb_net.fc.in_features  
        self.rgb_net.fc = nn.Identity()
        self.flow_net.fc = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 2, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(128, 2)
        )
    def forward(self, rgb, flow):
        f1 = self.rgb_net(rgb)
        f2 = self.flow_net(flow)
        return self.head(torch.cat([f1, f2], dim=1))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerClassifier(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=128, nhead=8, num_layers=3):
        super().__init__()
        self.embed = nn.Linear(input_dim, hidden_dim)
        self.pos_enc = PositionalEncoding(hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=512,
            batch_first=True, dropout=0.1, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(hidden_dim, 2)
    def forward(self, x, mask=None):
        x = self.embed(x)
        x = self.pos_enc(x)
        x = self.transformer(x, src_key_padding_mask=~mask if mask is not None else None)
        if mask is not None:
            x = (x * mask.unsqueeze(-1)).sum(1) / (mask.sum(1, keepdims=True) + 1e-8)
        else:
            x = x.mean(1)
        return self.fc(x)

def load_model_safe(path, model):
    ckpt = torch.load(path, map_location=DEVICE)
    state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict)
    return model.eval().to(DEVICE)

modelA = load_model_safe(modelA_path, TwoStreamI3D_Fast())
modelB = load_model_safe(modelB_path, TransformerClassifier())
print("Models loaded successfully!")

# -------------------------------
# YOLO + DeepSORT 
# -------------------------------
yolo = YOLO("yolov8n.pt")
tracker = DeepSort(max_age=30, embedder_gpu=True)

def extract_frames_fast(video_path, skip=16):
    cap = cv2.VideoCapture(video_path)
    frames = []
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if count % skip == 0:
            frames.append(frame)
        count += 1
    cap.release()
    return frames

def detect_and_track(frames):
    trajectories = {}
    for t, frame in enumerate(frames):
        results = yolo(frame, verbose=False)[0]
        dets = []
        if results.boxes is not None:
            for box, conf, cls_id in zip(results.boxes.xyxy, results.boxes.conf, results.boxes.cls):
                if int(cls_id) in [2, 3, 5, 7] and conf > 0.4:
                    bbox = box.cpu().tolist()  # [x1, y1, x2, y2]
                    dets.append([bbox, float(conf), int(cls_id)])  # ← CORRECT FORMAT
        tracks = tracker.update_tracks(dets, frame=frame)
        for tr in tracks:
            if not tr.is_confirmed(): continue
            tid = tr.track_id
            if tid not in trajectories: trajectories[tid] = []
            x1, y1, x2, y2 = tr.to_tlbr()
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            trajectories[tid].append([t, cx, cy])
    return trajectories

def compute_kinematics(trajectories):
    feats = {}
    for tid, pts in trajectories.items():
        pts = np.array(pts, dtype=np.float32)
        if len(pts) < 5: continue
        pos = pts[:, 1:]
        vel = np.diff(pos, axis=0, prepend=pos[:1])
        speed = np.linalg.norm(vel, axis=1, keepdims=True)
        accel = np.diff(speed, axis=0, prepend=speed[:1])
        seq = np.hstack([pos, speed, accel])
        seq = (seq - seq.mean(0)) / (seq.std(0) + 1e-8)
        feats[tid] = seq
    return feats

# -------------------------------
# Model A utils
# -------------------------------
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def compute_flow_clip(frames):
    flows = []
    prev = None
    for f in frames:
        gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        if prev is None:
            flows.append(np.zeros_like(f))
            prev = gray
            continue
        flow = cv2.calcOpticalFlowFarneback(prev, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
        hsv = np.zeros_like(f)
        hsv[...,0] = ang * 180 / np.pi / 2
        hsv[...,1] = 255
        hsv[...,2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
        flows.append(cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR))
        prev = gray
    return flows

# -------------------------------
# Model inference
# -------------------------------
@torch.no_grad()
def predict_video_modelA(video_path):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if total_frames < SEQ_LEN: return 0.0

    probs = []
    for _ in range(MAX_CLIPS_A):
        start = random.randint(0, max(1, total_frames - SEQ_LEN))
        cap = cv2.VideoCapture(video_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        clip = []
        for _ in range(SEQ_LEN):
            ret, frame = cap.read()
            if not ret: break
            clip.append(frame)
        cap.release()
        if len(clip) < SEQ_LEN: continue

        rgb = torch.stack([transform(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in clip]).unsqueeze(0).to(DEVICE)
        flow = torch.stack([transform(f) for f in compute_flow_clip(clip)]).unsqueeze(0).to(DEVICE)
        rgb = rgb.permute(0, 2, 1, 3, 4)
        flow = flow.permute(0, 2, 1, 3, 4)
        prob = softmax(modelA(rgb, flow), dim=1)[0, 1].item()
        probs.append(prob)

    return float(np.mean(probs)) if probs else 0.0

@torch.no_grad()
def predict_video_modelB(video_path):
    frames = extract_frames_fast(video_path, skip=16)
    if len(frames) < 8: return 0.0
    trajectories = detect_and_track(frames)
    feats = compute_kinematics(trajectories)
    if not feats: return 0.0
    seqs = list(feats.values())
    max_len = max(len(s) for s in seqs)
    padded = [np.pad(s, ((0, max_len - len(s)), (0, 0))) for s in seqs]
    x = torch.tensor(np.array(padded), dtype=torch.float32).to(DEVICE)
    mask = torch.zeros(len(seqs), max_len, dtype=torch.bool, device=DEVICE)
    for i, s in enumerate(seqs):
        mask[i, :len(s)] = True
    logits = modelB(x, mask)
    return float(softmax(logits, dim=1)[:, 1].cpu().numpy().mean())

# -------------------------------
# Loading 5+5 videos
# -------------------------------
accident_videos = []
normal_videos = []

for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if not f.lower().endswith('.mp4'): continue
        path = os.path.join(root, f)
        name = (os.path.basename(root) + f).lower()
        if any(k in name for k in ["accident", "crash", "positive", "incident", "1"]):
            accident_videos.append(path)
        else:
            normal_videos.append(path)

videos = accident_videos[:5] + normal_videos[:5]
labels = [1] * len(accident_videos[:5]) + [0] * len(normal_videos[:5])

print(f"\nQuick evaluation on {len(videos)} videos (5 accident + 5 normal)")

# -------------------------------
# Run inference
# -------------------------------
results = []
wA, wB = 0.55, 0.45

for video_path, true_label in tqdm(zip(videos, labels), total=len(videos)):
    probA = predict_video_modelA(video_path)
    probB = predict_video_modelB(video_path)
    fusion_prob = wA * probA + wB * probB
    pred = 1 if fusion_prob > 0.5 else 0
    
    results.append({
        "video": os.path.basename(video_path),
        "true": true_label,
        "probA": round(probA, 3),
        "probB": round(probB, 3),
        "fusion": round(fusion_prob, 3),
        "pred": pred
    })
    
    torch.cuda.empty_cache()
    gc.collect()

df = pd.DataFrame(results)
y_true = df["true"].values
y_pred = df["pred"].values

print("\n" + "="*70)
print("="*70)
print(f"Accuracy : {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.3f}")x
print(f"F1-score : {f1_score(y_true, y_pred, zero_division=0):.3f}")
print("\nPredictions:")
print(df[["video", "true", "probA", "probB", "fusion", "pred"]].to_string(index=False))

Using device: cuda
Models loaded successfully!

Quick evaluation on 10 videos (5 accident + 5 normal)


  0%|          | 0/10 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
100%|██████████| 10/10 [06:29<00:00, 38.99s/it]


QUICK EVALUATION RESULTS (10 videos)
Accuracy : 0.600
Precision: 0.571
Recall   : 0.800
F1-score : 0.667

Predictions:
             video  true  probA  probB  fusion  pred
other_crash_44.mp4     1  0.466  1.000   0.706     1
 other_crash_6.mp4     1  0.975  0.809   0.901     1
other_crash_32.mp4     1  0.841  0.000   0.463     0
other_crash_45.mp4     1  0.593  0.670   0.628     1
 other_crash_2.mp4     1  0.999  0.000   0.550     1
   rollover_99.mp4     0  0.368  0.747   0.538     1
   rollover_87.mp4     0  0.026  0.845   0.395     0
   rollover_52.mp4     0  0.746  0.725   0.737     1
   rollover_94.mp4     0  0.833  0.720   0.782     1
   rollover_77.mp4     0  0.512  0.000   0.282     0
